# Student Performance Decision-Support System

## Project Foundation Notebook

This notebook establishes the shared dataset, target definition, feature sets, data-quality checks, and train-test split used throughout the project.

In [41]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

# Find the repository root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Allow imports from the src folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("src folder exists:", (PROJECT_ROOT / "src").exists())

from src.config import (
    RANDOM_STATE,
    TEST_SIZE,
    TARGET_COLUMN,
    RAW_DATA_PATH,
    TRAIN_INDICES_PATH,
    TEST_INDICES_PATH,
)

from src.data import (
    load_student_data,
    create_target,
    get_feature_sets,
    save_split_indices,
)

print("Project imports completed successfully.")

Current directory: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/notebooks
Project root: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support
src folder exists: True
Project imports completed successfully.


In [42]:
# Load the raw dataset
df_raw = load_student_data()

print("Dataset path:", RAW_DATA_PATH)
print("Dataset shape:", df_raw.shape)

df_raw.head()

Dataset path: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/data/raw/student-por.csv
Dataset shape: (649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [43]:
# Display all column names in the dataset
print("Dataset columns:")
print(df_raw.columns.tolist())

# Display the data type assigned to each column
print("\nData types:")
display(df_raw.dtypes.to_frame(name="data_type"))

Dataset columns:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

Data types:


,data_type
school,str
sex,str
age,int64
address,str
famsize,str
Pstatus,str
Medu,int64
Fedu,int64
Mjob,str
Fjob,str


In [44]:
# Show a summary of the dataset structure
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      649 non-null    str  
 1   sex         649 non-null    str  
 2   age         649 non-null    int64
 3   address     649 non-null    str  
 4   famsize     649 non-null    str  
 5   Pstatus     649 non-null    str  
 6   Medu        649 non-null    int64
 7   Fedu        649 non-null    int64
 8   Mjob        649 non-null    str  
 9   Fjob        649 non-null    str  
 10  reason      649 non-null    str  
 11  guardian    649 non-null    str  
 12  traveltime  649 non-null    int64
 13  studytime   649 non-null    int64
 14  failures    649 non-null    int64
 15  schoolsup   649 non-null    str  
 16  famsup      649 non-null    str  
 17  paid        649 non-null    str  
 18  activities  649 non-null    str  
 19  nursery     649 non-null    str  
 20  higher      649 non-null    str  
 21  inte

In [45]:
# Count missing values in each column
missing_values = df_raw.isnull().sum()

# Keep only columns that contain at least one missing value
missing_summary = (
    missing_values[missing_values > 0]
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

# Report whether missing values were found
if missing_summary.empty:
    print("No missing values were found.")
else:
    display(missing_summary)

No missing values were found.


In [46]:
# Count rows that are exact duplicates of earlier rows
duplicate_count = df_raw.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


In [47]:
# Create the binary target column
df = create_target(df_raw)

# Preview the final grade and its corresponding target value
df[["G3", TARGET_COLUMN]].head(10)

,G3,needs_support
0,11,0
1,11,0
2,12,0
3,14,0
4,13,0
5,13,0
6,13,0
7,13,0
8,17,0
9,13,0


In [48]:
# Independently recreate the expected target values
expected_target = (df["G3"] < 10).astype(int)

# Check whether every generated target value follows the agreed rule
target_rule_correct = (
    df[TARGET_COLUMN] == expected_target
).all()


In [49]:
# Count the number of students in each target class
class_counts = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

# Calculate the percentage of students in each target class
class_percentages = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# Combine counts and percentages into one table
class_balance = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages,
})

# Replace numeric class labels with readable descriptions
class_balance.index = [
    "Likely to pass",
    "May require academic support",
]

display(class_balance)

,count,percentage
Likely to pass,549,84.59
May require academic support,100,15.41


In [50]:
# Create the shared feature sets and target:
# X_base contains all approved features except G3 and the target.
# X_early excludes G1 and G2.
# X_progress includes G1 and G2.
# y contains the binary target.
X_base, X_early, X_progress, y = get_feature_sets(df)

# Display the dimensions of each dataset.
print("Base feature shape:", X_base.shape)
print("Early-warning feature shape:", X_early.shape)
print("Progress-informed feature shape:", X_progress.shape)
print("Target shape:", y.shape)

Base feature shape: (649, 32)
Early-warning feature shape: (649, 30)
Progress-informed feature shape: (649, 32)
Target shape: (649,)


In [51]:
# Confirm that G3 is not used as an input feature.
assert "G3" not in X_base.columns
assert "G3" not in X_early.columns
assert "G3" not in X_progress.columns

# Confirm that the early-warning model excludes G1 and G2.
assert "G1" not in X_early.columns
assert "G2" not in X_early.columns

# Confirm that the progress-informed model includes G1 and G2.
assert "G1" in X_progress.columns
assert "G2" in X_progress.columns

# Confirm that the target column is not included in any feature set.
assert TARGET_COLUMN not in X_base.columns
assert TARGET_COLUMN not in X_early.columns
assert TARGET_COLUMN not in X_progress.columns


In [52]:
# Identify columns stored as numerical data.
numeric_features = (
    X_base
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

# Identify columns stored as categorical or text data.
categorical_features = (
    X_base
    .select_dtypes(include=["object", "category", "bool"])
    .columns
    .tolist()
)

# Display the number of features in each group.
print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

# Display the feature names.
print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 15
Number of categorical features: 17

Numerical features:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']

Categorical features:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


/var/folders/8v/3__g_c6x1rd5b66tsc2jd4lw0000gn/T/ipykernel_44122/1062602847.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category", "bool"])


In [53]:
all_indices = df.index.to_numpy()

# Create a stratified train-test split.
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Display the number of rows assigned to each set.
print("Training rows:", len(train_indices))
print("Testing rows:", len(test_indices))

Training rows: 519
Testing rows: 130


In [54]:
# Calculate the target distribution in the full dataset.
overall_distribution = (
    y.value_counts(normalize=True)
    .sort_index()
)

# Calculate the target distribution in the training set.
train_distribution = (
    y.loc[train_indices]
    .value_counts(normalize=True)
    .sort_index()
)

# Calculate the target distribution in the testing set.
test_distribution = (
    y.loc[test_indices]
    .value_counts(normalize=True)
    .sort_index()
)

# Combine the distributions for comparison.
distribution_check = pd.DataFrame({
    "overall": overall_distribution,
    "train": train_distribution,
    "test": test_distribution,
}).round(4)

# Add readable class names.
distribution_check.index = [
    "Likely to pass",
    "May require academic support",
]

display(distribution_check)

,overall,train,test
Likely to pass,0.8459,0.8459,0.8462
May require academic support,0.1541,0.1541,0.1538


In [55]:
# Save the shared row indices so every teammate and model
save_split_indices(
    train_indices=train_indices,
    test_indices=test_indices,
)

# Display where the files were saved
print("Training indices saved to:", TRAIN_INDICES_PATH)
print("Testing indices saved to:", TEST_INDICES_PATH)


Training indices saved to: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/data/processed/train_indices.csv
Testing indices saved to: /Users/jamal/Documents/2nd Year/2nd_Sem/CS254_Introduction_to_Artificial_Intelligence/Final Project/student-performance-decision-support/data/processed/test_indices.csv


In [56]:
# Confirm that every row belongs to either the training or testing set.
assert len(train_indices) + len(test_indices) == len(df)

# Confirm that no row appears in both sets.
assert set(train_indices).isdisjoint(set(test_indices))

# Confirm that the saved index files exist.
assert TRAIN_INDICES_PATH.exists()
assert TEST_INDICES_PATH.exists()


## Foundation Summary

- The Portuguese student-performance dataset was loaded successfully.
- The dataset contains 649 student records and 33 original columns.
- The binary target is `needs_support`.
- `needs_support = 1` means the student may require academic support.
- `needs_support = 0` means the student is likely to pass.
- `G3` is excluded from all model inputs.
- The early-warning experiment excludes `G1` and `G2`.
- The progress-informed experiment includes `G1` and `G2`.
- Numerical and categorical features have been identified.
- A shared stratified train-test split has been created.
- The shared train and test row indices have been saved for all team members.